# Holiday Planner, simplified integrated API pipeline

This version keeps the outputs simple and useful for the rest of the team.

APIs used:
- Google Geocoding API for coordinates and country
- OpenWeather forecast API for weather statistics over the available forecast range
- Google Places API (New) for restaurants, bars, nightclubs, lodging and attractions
- SerpApi to webscrape Google Flights for flight price estimates
- SerpApi to webscrape Google Hotels for one selected accommodation option per destination

Main outputs:
- `destination_recommendations.csv`, one row per destination
- `places.csv`, one row per nearby amenity
- `weather_daily.csv`, one row per destination per forecast day
- `selected_travel_options.csv`, one selected flight and one selected accommodation per destination

Destinations assessed:
- Top global holiday destinations from https://www.tripadvisor.co.uk/TravelersChoice-Destinations
- 24 destinations taken and London excluded as London is treated as city of origin

## 1. Setup

Create a `.env` file in the same folder as this notebook:

```bash
GOOGLE_MAPS_API_KEY=your_google_key_here
OPENWEATHER_API_KEY=your_openweather_key_here
SERPAPI_API_KEY=your_serpapi_key_here
```

Install packages once if needed:

```bash
pip install requests pandas python-dotenv
```

In [72]:
# Uncomment and run if needed
# !pip install requests pandas python-dotenv

In [73]:
import os
import re
import time
import json
from pathlib import Path
from datetime import date, datetime, timedelta

import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv()

GOOGLE_MAPS_API_KEY = os.getenv("GOOGLE_MAPS_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")

missing = [
    key for key, value in {
        "GOOGLE_MAPS_API_KEY": GOOGLE_MAPS_API_KEY,
        "OPENWEATHER_API_KEY": OPENWEATHER_API_KEY,
        "SERPAPI_API_KEY": SERPAPI_API_KEY,
    }.items() if not value
]

if missing:
    raise ValueError("Missing API key(s): " + ", ".join(missing))

print("API keys loaded.")

API keys loaded.


## 2. Project settings

Edit destinations, airport codes and dates here.

Weather note: OpenWeather's standard forecast endpoint gives a short forecast window. The weather statistics below describe the available forecast window, while the SerpApi dates are used for flights and accommodation.

In [75]:
DESTINATIONS = [
    "Bali, Indonesia",
    "Dubai, United Arab Emirates",
    "Hanoi, Vietnam",
    "Paris, France",
    "Rome, Italy",
    "Marrakech, Morocco",
    "Bangkok, Thailand",
    "Crete, Greece",
    "New York City, United States",
    "Siem Reap, Cambodia",
    "Istanbul, Türkiye",
    "Cusco, Peru",
    "Barcelona, Spain",
    "Lisbon, Portugal",
    "Tokyo, Japan",
    "Kathmandu, Nepal",
    "Edinburgh, United Kingdom",
    "Hurghada, Egypt",
    "Cabo San Lucas, Mexico",
    "Maldives, Maldives",
    "New Delhi, India",
    "Budapest, Hungary",
    "Seoul, South Korea",
    "Abu Dhabi, United Arab Emirates",
]

PLACE_TYPES = ["restaurant", "bar", "night_club", "lodging", "tourist_attraction"]
PLACES_RADIUS_METRES = 5000
MAX_PLACES_PER_TYPE = 20

ORIGIN_AIRPORT_CODE = "LHR,LGW,STN,LTN,LCY"
ADULTS = 1
CURRENCY = "GBP"

DEPARTURE_DATE = (date.today() + timedelta(days=60)).isoformat()
RETURN_DATE = (date.today() + timedelta(days=67)).isoformat()
TRIP_NIGHTS = max(1, (date.fromisoformat(RETURN_DATE) - date.fromisoformat(DEPARTURE_DATE)).days)

DESTINATION_AIRPORT_CODES = {
    "Bali, Indonesia": "DPS",
    "Dubai, United Arab Emirates": "DXB",
    "Hanoi, Vietnam": "HAN",
    "Paris, France": "CDG",
    "Rome, Italy": "FCO",
    "Marrakech, Morocco": "RAK",
    "Bangkok, Thailand": "BKK",
    "Crete, Greece": "HER",
    "New York City, United States": "JFK",
    "Siem Reap, Cambodia": "SAI",
    "Istanbul, Türkiye": "IST",
    "Cusco, Peru": "CUZ",
    "Barcelona, Spain": "BCN",
    "Lisbon, Portugal": "LIS",
    "Tokyo, Japan": "HND",
    "Kathmandu, Nepal": "KTM",
    "Edinburgh, United Kingdom": "EDI",
    "Hurghada, Egypt": "HRG",
    "Cabo San Lucas, Mexico": "SJD",
    "Maldives, Maldives": "MLE",
    "New Delhi, India": "DEL",
    "Budapest, Hungary": "BUD",
    "Seoul, South Korea": "ICN",
    "Abu Dhabi, United Arab Emirates": "AUH",
}

MAX_FLIGHTS_PER_DESTINATION = 10
MAX_HOTELS_PER_DESTINATION = 20
MAX_RENTALS_PER_DESTINATION = 20

MIN_ACCOMMODATION_NIGHTLY_PRICE = 20
MAX_ACCOMMODATION_NIGHTLY_PRICE = 1000
EXCLUDED_ACCOMMODATION_TERMS = ["hostel", "hostal", "dorm", "camping", "campsite", "shared room", "bed in", "backpacker"]

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Trip: {DEPARTURE_DATE} to {RETURN_DATE} ({TRIP_NIGHTS} nights)")
print(f"Origin: {ORIGIN_AIRPORT_CODE}, currency: {CURRENCY}")

Trip: 2026-08-25 to 2026-09-01 (7 nights)
Origin: LHR,LGW,STN,LTN,LCY, currency: GBP


## 3. Helper functions

In [77]:
def slugify(value):
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9]+", "_", value).strip("_")
    return value or "unknown"


def save_raw_json(data, filename):
    (RAW_DIR / filename).write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def request_json(method, url, *, params=None, headers=None, json_body=None, timeout=30, retries=2):
    last_error = None
    for attempt in range(retries + 1):
        try:
            response = requests.request(method, url, params=params, headers=headers, json=json_body, timeout=timeout)
            if not response.ok:
                try:
                    detail = response.json()
                except ValueError:
                    detail = response.text
                raise requests.exceptions.HTTPError(f"HTTP {response.status_code}: {detail}", response=response)
            return response.json()
        except requests.exceptions.RequestException as error:
            last_error = error
            print(f"Request failed on attempt {attempt + 1}: {error}")
            if attempt < retries:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"API request failed after {retries + 1} attempts: {last_error}")


def parse_price(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, dict):
        for key in ["extracted_lowest", "extracted_price", "extracted_total", "extracted_before_taxes_fees", "value", "lowest", "price", "total", "before_taxes_fees"]:
            if key in value and value[key] is not None:
                return parse_price(value[key])
        return None
    if isinstance(value, str):
        cleaned = re.sub(r"[^0-9.]", "", value.replace(",", ""))
        try:
            return float(cleaned) if cleaned else None
        except ValueError:
            return None
    return None


def first_mode(series):
    values = series.dropna()
    if values.empty:
        return None
    modes = values.mode()
    return modes.iloc[0] if not modes.empty else values.iloc[0]

## 4. Location, weather and places APIs

In [79]:
def geocode_destination(destination):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": destination, "key": GOOGLE_MAPS_API_KEY}
    data = request_json("GET", url, params=params)
    save_raw_json(data, f"google_geocode_{slugify(destination)}.json")

    if data.get("status") != "OK":
        raise ValueError(f"Geocoding failed for {destination}: {data}")

    result = data["results"][0]
    location = result["geometry"]["location"]
    country = None
    for component in result.get("address_components", []):
        if "country" in component.get("types", []):
            country = component.get("long_name")
            break

    return {
        "destination": destination,
        "formatted_address": result.get("formatted_address"),
        "country": country,
        "latitude": location.get("lat"),
        "longitude": location.get("lng"),
        "google_geocode_place_id": result.get("place_id"),
    }


def get_weather_forecast(destination, latitude, longitude):
    url = "https://api.openweathermap.org/data/2.5/forecast"
    params = {"lat": latitude, "lon": longitude, "appid": OPENWEATHER_API_KEY, "units": "metric"}
    data = request_json("GET", url, params=params)
    save_raw_json(data, f"openweather_forecast_{slugify(destination)}.json")

    rows = []
    for item in data.get("list", []):
        dt_txt = item.get("dt_txt")
        desc = item.get("weather", [{}])[0].get("description") if item.get("weather") else None
        rows.append({
            "destination": destination,
            "forecast_datetime": dt_txt,
            "forecast_date": dt_txt[:10] if dt_txt else None,
            "temp_c": item.get("main", {}).get("temp"),
            "max_temp_c": item.get("main", {}).get("temp_max"),
            "humidity_pct": item.get("main", {}).get("humidity"),
            "cloudiness_pct": item.get("clouds", {}).get("all"),
            "wind_speed_mps": item.get("wind", {}).get("speed"),
            "weather_description": desc,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    return (
        df.groupby(["destination", "forecast_date"], as_index=False)
        .agg(
            daily_avg_temp_c=("temp_c", "mean"),
            daily_min_temp_c=("temp_c", "min"),
            daily_max_temp_c=("max_temp_c", "max"),
            daily_avg_humidity_pct=("humidity_pct", "mean"),
            daily_avg_cloudiness_pct=("cloudiness_pct", "mean"),
            daily_avg_wind_speed_mps=("wind_speed_mps", "mean"),
            daily_max_wind_speed_mps=("wind_speed_mps", "max"),
            most_common_description=("weather_description", first_mode),
        )
    )


def create_weather_summary(weather_daily_df, destinations):
    base = pd.DataFrame({"destination": destinations})
    if weather_daily_df.empty:
        return base

    summary = (
        weather_daily_df.groupby("destination", as_index=False)
        .agg(
            forecast_start_date=("forecast_date", "min"),
            forecast_end_date=("forecast_date", "max"),
            forecast_days=("forecast_date", "nunique"),
            avg_temp_c=("daily_avg_temp_c", "mean"),
            min_daily_temp_c=("daily_min_temp_c", "min"),
            peak_max_temp_c=("daily_max_temp_c", "max"),
            avg_humidity_pct=("daily_avg_humidity_pct", "mean"),
            avg_cloudiness_pct=("daily_avg_cloudiness_pct", "mean"),
            avg_wind_speed_mps=("daily_avg_wind_speed_mps", "mean"),
            max_wind_speed_mps=("daily_max_wind_speed_mps", "max"),
            dominant_weather_description=("most_common_description", first_mode),
        )
    )

    for col in ["avg_temp_c", "min_daily_temp_c", "peak_max_temp_c", "avg_humidity_pct", "avg_cloudiness_pct", "avg_wind_speed_mps", "max_wind_speed_mps"]:
        summary[col] = summary[col].round(1)

    return base.merge(summary, on="destination", how="left")


def get_nearby_places(destination, latitude, longitude, place_type):
    url = "https://places.googleapis.com/v1/places:searchNearby"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_MAPS_API_KEY,
        "X-Goog-FieldMask": "places.id,places.displayName,places.formattedAddress,places.location,places.rating,places.userRatingCount,places.priceLevel,places.types,places.googleMapsUri",
    }
    body = {
        "includedTypes": [place_type],
        "maxResultCount": MAX_PLACES_PER_TYPE,
        "locationRestriction": {"circle": {"center": {"latitude": latitude, "longitude": longitude}, "radius": PLACES_RADIUS_METRES}},
    }
    data = request_json("POST", url, headers=headers, json_body=body)
    save_raw_json(data, f"google_places_{slugify(destination)}_{slugify(place_type)}.json")

    rows = []
    for place in data.get("places", []):
        loc = place.get("location", {})
        rows.append({
            "destination": destination,
            "place_type_searched": place_type,
            "place_id": place.get("id"),
            "place_name": place.get("displayName", {}).get("text"),
            "formatted_address": place.get("formattedAddress"),
            "latitude": loc.get("latitude"),
            "longitude": loc.get("longitude"),
            "rating": place.get("rating"),
            "user_rating_count": place.get("userRatingCount"),
            "price_level": place.get("priceLevel"),
            "google_maps_url": place.get("googleMapsUri"),
            "types": ", ".join(place.get("types", [])) if place.get("types") else None,
        })
    return pd.DataFrame(rows)


def create_places_summary(places_df, destinations):
    base = pd.DataFrame({"destination": destinations})
    expected = ["restaurants_count", "bars_count", "nightclubs_count", "hotels_count", "attractions_count"]
    if places_df.empty:
        for col in expected:
            base[col] = 0
        return base

    counts = (
        places_df.groupby(["destination", "place_type_searched"])
        .size().reset_index(name="count")
        .pivot(index="destination", columns="place_type_searched", values="count")
        .fillna(0).reset_index()
        .rename(columns={"restaurant": "restaurants_count", "bar": "bars_count", "night_club": "nightclubs_count", "lodging": "hotels_count", "tourist_attraction": "attractions_count"})
    )
    out = base.merge(counts, on="destination", how="left")
    for col in expected:
        if col not in out.columns:
            out[col] = 0
        out[col] = out[col].fillna(0).astype(int)
    return out

## 5. SerpApi flights and accommodation

In [81]:
def serpapi_search(params, raw_filename):
    params = dict(params)
    params["api_key"] = SERPAPI_API_KEY
    data = request_json("GET", "https://serpapi.com/search", params=params, timeout=60, retries=1)
    save_raw_json(data, raw_filename)
    if "error" in data:
        raise ValueError(data["error"])
    return data


def get_serpapi_flight_prices(destination, arrival_airport_code):
    if not arrival_airport_code:
        return pd.DataFrame()
    params = {
        "engine": "google_flights",
        "departure_id": ORIGIN_AIRPORT_CODE,
        "arrival_id": arrival_airport_code,
        "outbound_date": DEPARTURE_DATE,
        "return_date": RETURN_DATE,
        "currency": CURRENCY,
        "hl": "en",
        "gl": "uk",
        "type": "1",
    }
    try:
        data = serpapi_search(params, f"serpapi_flights_{slugify(destination)}.json")
    except Exception as error:
        print(f"Flight price search failed for {destination}: {error}")
        return pd.DataFrame()

    rows = []
    insights = data.get("price_insights", {}) or {}
    items = []
    for result_type in ["best_flights", "other_flights"]:
        for item in data.get(result_type, []) or []:
            items.append((result_type, item))

    for result_type, item in items[:MAX_FLIGHTS_PER_DESTINATION]:
        legs = item.get("flights", []) or []
        first = legs[0] if legs else {}
        last = legs[-1] if legs else {}
        rows.append({
            "destination": destination,
            "origin_airport_code": ORIGIN_AIRPORT_CODE,
            "arrival_airport_code": arrival_airport_code,
            "departure_date": DEPARTURE_DATE,
            "return_date": RETURN_DATE,
            "currency": CURRENCY,
            "airline": first.get("airline"),
            "departure_airport": first.get("departure_airport", {}).get("name"),
            "arrival_airport": last.get("arrival_airport", {}).get("name"),
            "total_duration_minutes": item.get("total_duration"),
            "stops": max(0, len(legs) - 1),
            "flight_price": parse_price(item.get("price")),
            "price_level": insights.get("price_level"),
        })
    return pd.DataFrame(rows)


def get_serpapi_hotel_prices(destination):
    params = {
        "engine": "google_hotels",
        "q": f"3 star hotels in {destination}",
        "check_in_date": DEPARTURE_DATE,
        "check_out_date": RETURN_DATE,
        "adults": ADULTS,
        "currency": CURRENCY,
        "hl": "en",
        "gl": "uk",
    }
    try:
        data = serpapi_search(params, f"serpapi_hotels_{slugify(destination)}.json")
    except Exception as error:
        print(f"Hotel price search failed for {destination}: {error}")
        return pd.DataFrame()
    return parse_serpapi_accommodation(destination, data, "hotel", MAX_HOTELS_PER_DESTINATION)


def get_serpapi_rental_prices(destination):
    params = {
        "engine": "google_hotels",
        "q": f"holiday apartments in {destination}",
        "check_in_date": DEPARTURE_DATE,
        "check_out_date": RETURN_DATE,
        "adults": ADULTS,
        "currency": CURRENCY,
        "hl": "en",
        "gl": "uk",
        "vacation_rentals": "true",
    }
    try:
        data = serpapi_search(params, f"serpapi_vacation_rentals_{slugify(destination)}.json")
    except Exception as error:
        print(f"Vacation rental search failed for {destination}: {error}")
        return pd.DataFrame()
    return parse_serpapi_accommodation(destination, data, "rental", MAX_RENTALS_PER_DESTINATION)


def parse_serpapi_accommodation(destination, data, accommodation_type, max_rows):
    rows = []
    for prop in (data.get("properties", []) or [])[:max_rows]:
        gps = prop.get("gps_coordinates", {}) or {}
        rows.append({
            "destination": destination,
            "accommodation_type": accommodation_type,
            "accommodation_name": prop.get("name"),
            "description": prop.get("description"),
            "overall_rating": prop.get("overall_rating"),
            "reviews": prop.get("reviews"),
            "latitude": gps.get("latitude"),
            "longitude": gps.get("longitude"),
            "nightly_price": parse_price(prop.get("rate_per_night")),
            "total_price": parse_price(prop.get("total_rate")),
            "source": prop.get("source"),
            "link": prop.get("link"),
        })
    return pd.DataFrame(rows)

## 6. Select one travel option per destination

In [83]:
def select_best_flight(flight_prices_df, destinations):
    base = pd.DataFrame({"destination": destinations})
    cols = ["destination", "selected_flight_price", "selected_flight_currency", "selected_airline", "selected_flight_stops", "selected_flight_duration_minutes", "selected_flight_data_status"]
    if flight_prices_df.empty or "flight_price" not in flight_prices_df:
        base["selected_flight_data_status"] = "no flight results"
        for c in cols:
            if c not in base:
                base[c] = pd.NA
        return base[cols]
    valid = flight_prices_df.dropna(subset=["flight_price"]).sort_values(["destination", "flight_price", "stops"])
    if valid.empty:
        base["selected_flight_data_status"] = "no usable flight price"
        for c in cols:
            if c not in base:
                base[c] = pd.NA
        return base[cols]
    selected = valid.groupby("destination", as_index=False).first().rename(columns={
        "flight_price": "selected_flight_price",
        "currency": "selected_flight_currency",
        "airline": "selected_airline",
        "stops": "selected_flight_stops",
        "total_duration_minutes": "selected_flight_duration_minutes",
    })
    selected["selected_flight_data_status"] = "flight available"
    out = base.merge(selected[cols], on="destination", how="left")
    out["selected_flight_data_status"] = out["selected_flight_data_status"].fillna("no flight results")
    return out[cols]


def filter_sensible_accommodation(df):
    if df.empty:
        return df
    df = df.copy()
    df["accommodation_name"] = df.get("accommodation_name", pd.Series(dtype=str)).fillna("")
    df["description"] = df.get("description", pd.Series(dtype=str)).fillna("")
    pattern = "|".join([re.escape(t) for t in EXCLUDED_ACCOMMODATION_TERMS])
    name_lower = df["accommodation_name"].str.lower()
    desc_lower = df["description"].str.lower()
    df = df[~name_lower.str.contains(pattern, regex=True, na=False) & ~desc_lower.str.contains(pattern, regex=True, na=False)].copy()
    df = df[df["nightly_price"].isna() | df["nightly_price"].between(MIN_ACCOMMODATION_NIGHTLY_PRICE, MAX_ACCOMMODATION_NIGHTLY_PRICE)].copy()
    df = df[df["total_price"].isna() | df["total_price"].between(MIN_ACCOMMODATION_NIGHTLY_PRICE * TRIP_NIGHTS, MAX_ACCOMMODATION_NIGHTLY_PRICE * TRIP_NIGHTS)].copy()
    df["total_price"] = df["total_price"].combine_first(df["nightly_price"] * TRIP_NIGHTS)
    return df


def select_best_accommodation(hotel_prices_df, rental_prices_df, destinations):
    base = pd.DataFrame({"destination": destinations})
    cols = ["destination", "selected_accommodation_type", "selected_accommodation_name", "selected_accommodation_nightly_price", "selected_accommodation_total_price", "selected_accommodation_rating", "selected_accommodation_reviews", "selected_accommodation_link", "selected_accommodation_data_status"]
    frames = [df for df in [hotel_prices_df, rental_prices_df] if not df.empty]
    if not frames:
        base["selected_accommodation_data_status"] = "no accommodation results"
        for c in cols:
            if c not in base:
                base[c] = pd.NA
        return base[cols]
    all_options = pd.concat(frames, ignore_index=True)
    valid = filter_sensible_accommodation(all_options)
    if valid.empty:
        base["selected_accommodation_data_status"] = "no sensible accommodation after filtering"
        for c in cols:
            if c not in base:
                base[c] = pd.NA
        return base[cols]
    valid = valid.sort_values(["destination", "total_price", "overall_rating", "reviews"], ascending=[True, True, False, False], na_position="last")
    selected = valid.groupby("destination", as_index=False).first().rename(columns={
        "accommodation_type": "selected_accommodation_type",
        "accommodation_name": "selected_accommodation_name",
        "nightly_price": "selected_accommodation_nightly_price",
        "total_price": "selected_accommodation_total_price",
        "overall_rating": "selected_accommodation_rating",
        "reviews": "selected_accommodation_reviews",
        "link": "selected_accommodation_link",
    })
    selected["selected_accommodation_data_status"] = "accommodation available"
    out = base.merge(selected[cols], on="destination", how="left")
    out["selected_accommodation_data_status"] = out["selected_accommodation_data_status"].fillna("no sensible accommodation after filtering")
    return out[cols]


def create_selected_travel_options(flight_prices_df, hotel_prices_df, rental_prices_df, destinations):
    flights = select_best_flight(flight_prices_df, destinations)
    accommodation = select_best_accommodation(hotel_prices_df, rental_prices_df, destinations)
    out = flights.merge(accommodation, on="destination", how="outer")
    out["estimated_accommodation_cost"] = out["selected_accommodation_total_price"]
    out["estimated_trip_cost"] = out["selected_flight_price"] + out["selected_accommodation_total_price"]
    out["cost_data_status"] = "missing cost data"
    out.loc[out["selected_flight_price"].notna() & out["selected_accommodation_total_price"].notna(), "cost_data_status"] = "flight and accommodation available"
    out.loc[out["selected_flight_price"].isna() & out["selected_accommodation_total_price"].notna(), "cost_data_status"] = "accommodation only"
    out.loc[out["selected_flight_price"].notna() & out["selected_accommodation_total_price"].isna(), "cost_data_status"] = "flight only"
    return out

## 7. Scores and final recommendations

In [85]:
def score_weather_forecast(avg_temp_c, peak_max_temp_c, avg_humidity_pct, avg_cloudiness_pct, avg_wind_speed_mps, max_wind_speed_mps):
    score = 100
    if pd.notna(avg_temp_c):
        if avg_temp_c < 22:
            score -= 25
        elif avg_temp_c > 35:
            score -= 20
        elif 24 <= avg_temp_c <= 32:
            score += 5
    if pd.notna(peak_max_temp_c) and peak_max_temp_c > 38:
        score -= 15
    if pd.notna(avg_humidity_pct):
        if avg_humidity_pct > 85:
            score -= 15
        elif avg_humidity_pct < 40:
            score -= 5
    if pd.notna(avg_cloudiness_pct):
        if avg_cloudiness_pct > 75:
            score -= 15
        elif avg_cloudiness_pct < 40:
            score += 5
    if pd.notna(avg_wind_speed_mps):
        if avg_wind_speed_mps > 10:
            score -= 15
        elif avg_wind_speed_mps <= 6:
            score += 5
    if pd.notna(max_wind_speed_mps) and max_wind_speed_mps > 14:
        score -= 10
    return max(0, min(100, score))

def log_count_score(value, cap):
    value = max(0, value)
    return np.log1p(value) / np.log1p(cap)


def rating_score(value):
    if pd.isna(value):
        return 0.5
    return np.clip((value - 3.5) / 1.5, 0, 1)


def get_table_value(table, destination, column, default=0):
    try:
        value = table.loc[destination, column]
        if pd.isna(value):
            return default
        return value
    except Exception:
        return default


def build_place_scores(places_df, destinations=None, max_places_per_type=20):
    """
    Builds amenities and nightlife scores using:
    - number of places
    - average rating
    - number of highly rated places

    Output column names stay compatible with the existing Streamlit app.
    """
    if destinations is None:
        destinations = DESTINATIONS

    base = pd.DataFrame({"destination": destinations})

    output_cols = [
        "destination",
        "restaurants_count",
        "bars_count",
        "nightclubs_count",
        "hotels_count",
        "attractions_count",
        "amenity_score",
        "nightlife_score",
    ]

    if places_df.empty:
        for col in output_cols:
            if col != "destination":
                base[col] = 0
        return base[output_cols]

    df = places_df.copy()

    # Your notebook uses place_type_searched, not place_type
    if "place_type_searched" in df.columns:
        type_col = "place_type_searched"
    elif "place_type" in df.columns:
        type_col = "place_type"
    else:
        raise KeyError(
            "Could not find a place type column. Expected 'place_type_searched' or 'place_type'. "
            f"Available columns are: {list(df.columns)}"
        )

    df["rating"] = pd.to_numeric(df.get("rating", np.nan), errors="coerce")

    count_table = (
        df.groupby(["destination", type_col])
        .size()
        .unstack(fill_value=0)
    )

    rating_table = (
        df.groupby(["destination", type_col])["rating"]
        .mean()
        .unstack()
    )

    high_rated_df = df[df["rating"] >= 4.3]

    if high_rated_df.empty:
        high_rated_table = pd.DataFrame()
    else:
        high_rated_table = (
            high_rated_df.groupby(["destination", type_col])
            .size()
            .unstack(fill_value=0)
        )

    rows = []

    for destination in destinations:
        restaurant_count = get_table_value(count_table, destination, "restaurant", 0)
        bar_count = get_table_value(count_table, destination, "bar", 0)
        nightclub_count = get_table_value(count_table, destination, "night_club", 0)
        lodging_count = get_table_value(count_table, destination, "lodging", 0)
        attraction_count = get_table_value(count_table, destination, "tourist_attraction", 0)

        restaurant_rating = get_table_value(rating_table, destination, "restaurant", np.nan)
        bar_rating = get_table_value(rating_table, destination, "bar", np.nan)
        nightclub_rating = get_table_value(rating_table, destination, "night_club", np.nan)
        lodging_rating = get_table_value(rating_table, destination, "lodging", np.nan)
        attraction_rating = get_table_value(rating_table, destination, "tourist_attraction", np.nan)

        highly_rated_amenities = (
            get_table_value(high_rated_table, destination, "restaurant", 0)
            + get_table_value(high_rated_table, destination, "tourist_attraction", 0)
            + get_table_value(high_rated_table, destination, "lodging", 0)
        )

        highly_rated_nightlife = (
            get_table_value(high_rated_table, destination, "bar", 0)
            + get_table_value(high_rated_table, destination, "night_club", 0)
        )

        amenity_count_component = (
            0.40 * log_count_score(restaurant_count, max_places_per_type)
            + 0.40 * log_count_score(attraction_count, max_places_per_type)
            + 0.20 * log_count_score(lodging_count, max_places_per_type)
        )

        amenity_rating_component = np.mean([
            rating_score(restaurant_rating),
            rating_score(attraction_rating),
            rating_score(lodging_rating),
        ])

        amenity_high_rated_component = np.clip(highly_rated_amenities / 15, 0, 1)

        amenity_score = 100 * (
            0.55 * amenity_count_component
            + 0.30 * amenity_rating_component
            + 0.15 * amenity_high_rated_component
        )

        nightlife_count_component = (
            0.45 * log_count_score(bar_count, max_places_per_type)
            + 0.55 * log_count_score(nightclub_count, max_places_per_type)
        )

        nightlife_rating_component = np.mean([
            rating_score(bar_rating),
            rating_score(nightclub_rating),
        ])

        nightlife_high_rated_component = np.clip(highly_rated_nightlife / 10, 0, 1)

        nightlife_score = 100 * (
            0.60 * nightlife_count_component
            + 0.30 * nightlife_rating_component
            + 0.10 * nightlife_high_rated_component
        )

        rows.append({
            "destination": destination,
            "restaurants_count": int(restaurant_count),
            "bars_count": int(bar_count),
            "nightclubs_count": int(nightclub_count),
            "hotels_count": int(lodging_count),
            "attractions_count": int(attraction_count),
            "amenity_score": round(amenity_score, 1),
            "nightlife_score": round(nightlife_score, 1),
        })

    return pd.DataFrame(rows)[output_cols]

def add_cost_score(df):
    df = df.copy()
    valid = df["estimated_trip_cost"].dropna()
    if valid.empty:
        df["cost_score"] = pd.NA
        return df
    if valid.max() == valid.min():
        df["cost_score"] = 50
        return df
    df["cost_score"] = 100 - (100 * (df["estimated_trip_cost"] - valid.min()) / (valid.max() - valid.min()))
    df["cost_score"] = df["cost_score"].clip(0, 100).round(1)
    return df


def create_destination_recommendations(results, destinations):
    weather_summary = create_weather_summary(results["weather_daily"], destinations)
    
    place_scores = build_place_scores(
        results["places"],
        max_places_per_type=MAX_PLACES_PER_TYPE
    )
    
    selected_travel = create_selected_travel_options(
        results["flight_prices_raw"],
        results["hotel_prices_raw"],
        results["rental_prices_raw"],
        destinations
    )

    summary = (
        results["geocoded"]
        .merge(weather_summary, on="destination", how="left")
        .merge(place_scores, on="destination", how="left")
        .merge(selected_travel, on="destination", how="left")
    )

    for col in ["restaurants_count", "bars_count", "nightclubs_count", "hotels_count", "attractions_count"]:
        summary[col] = summary[col].fillna(0).astype(int)

    summary["weather_score"] = summary.apply(
        lambda r: score_weather_forecast(
            r.get("avg_temp_c"),
            r.get("peak_max_temp_c"),
            r.get("avg_humidity_pct"),
            r.get("avg_cloudiness_pct"),
            r.get("avg_wind_speed_mps"),
            r.get("max_wind_speed_mps")
        ),
        axis=1
    )

    summary = add_cost_score(summary)

    summary["score_without_cost"] = (
        0.40 * summary["weather_score"]
        + 0.35 * summary["nightlife_score"]
        + 0.25 * summary["amenity_score"]
    ).round(1)

    summary["cost_score_for_model"] = summary["cost_score"].fillna(50)

    summary["final_recommendation_score"] = (
        0.30 * summary["weather_score"]
        + 0.25 * summary["nightlife_score"]
        + 0.20 * summary["amenity_score"]
        + 0.25 * summary["cost_score_for_model"]
    ).round(1)

    return summary.sort_values("final_recommendation_score", ascending=False), selected_travel

## 8. Run pipeline

In [87]:
def run_pipeline(destinations):
    geocoded_rows, weather_daily_dfs, places_dfs = [], [], []
    flight_dfs, hotel_dfs, rental_dfs = [], [], []
    for destination in destinations:
        print("=" * 80)
        print(f"Processing: {destination}")
        geo = geocode_destination(destination)
        geocoded_rows.append(geo)
        lat, lon = geo["latitude"], geo["longitude"]

        weather = get_weather_forecast(destination, lat, lon)
        if not weather.empty:
            weather_daily_dfs.append(weather)

        for place_type in PLACE_TYPES:
            print(f"Places: {place_type}")
            places = get_nearby_places(destination, lat, lon, place_type)
            if not places.empty:
                places_dfs.append(places)
            time.sleep(0.2)

        flights = get_serpapi_flight_prices(destination, DESTINATION_AIRPORT_CODES.get(destination))
        if not flights.empty:
            flight_dfs.append(flights)
        time.sleep(0.5)

        hotels = get_serpapi_hotel_prices(destination)
        if not hotels.empty:
            hotel_dfs.append(hotels)
        time.sleep(0.5)

        rentals = get_serpapi_rental_prices(destination)
        if not rentals.empty:
            rental_dfs.append(rentals)
        time.sleep(0.5)

    return {
        "geocoded": pd.DataFrame(geocoded_rows),
        "weather_daily": pd.concat(weather_daily_dfs, ignore_index=True) if weather_daily_dfs else pd.DataFrame(),
        "places": pd.concat(places_dfs, ignore_index=True) if places_dfs else pd.DataFrame(),
        "flight_prices_raw": pd.concat(flight_dfs, ignore_index=True) if flight_dfs else pd.DataFrame(),
        "hotel_prices_raw": pd.concat(hotel_dfs, ignore_index=True) if hotel_dfs else pd.DataFrame(),
        "rental_prices_raw": pd.concat(rental_dfs, ignore_index=True) if rental_dfs else pd.DataFrame(),
    }

results = run_pipeline(DESTINATIONS)
destination_recommendations_df, selected_travel_options_df = create_destination_recommendations(results, DESTINATIONS)

print("Pipeline complete.")
for name, df in results.items():
    print(f"{name}: {len(df)} rows")
print(f"selected_travel_options: {len(selected_travel_options_df)} rows")
print(f"destination_recommendations: {len(destination_recommendations_df)} rows")

destination_recommendations_df.head()

Processing: Bali, Indonesia
Places: restaurant
Places: bar
Places: night_club
Places: lodging
Places: tourist_attraction
Processing: Dubai, United Arab Emirates
Places: restaurant
Places: bar
Places: night_club
Places: lodging
Places: tourist_attraction
Processing: Hanoi, Vietnam
Places: restaurant
Places: bar
Places: night_club
Places: lodging
Places: tourist_attraction
Processing: Paris, France
Places: restaurant
Places: bar
Places: night_club
Places: lodging
Places: tourist_attraction
Processing: Rome, Italy
Places: restaurant
Places: bar
Places: night_club
Places: lodging
Places: tourist_attraction
Processing: Marrakech, Morocco
Places: restaurant
Places: bar
Places: night_club
Places: lodging
Places: tourist_attraction
Processing: Bangkok, Thailand
Places: restaurant
Places: bar
Places: night_club
Places: lodging
Places: tourist_attraction
Processing: Crete, Greece
Places: restaurant
Places: bar
Places: night_club
Places: lodging
Places: tourist_attraction
Processing: New York Cit

C:\Users\fenne\AppData\Local\Temp\ipykernel_9556\2150994133.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  "places": pd.concat(places_dfs, ignore_index=True) if places_dfs else pd.DataFrame(),
C:\Users\fenne\AppData\Local\Temp\ipykernel_9556\2150994133.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  "rental_prices_raw": pd.concat(rental_dfs, ignore_index=True) if rental_dfs else pd.DataFrame(),


Pipeline complete.
geocoded: 24 rows
weather_daily: 144 rows
places: 2246 rows
flight_prices_raw: 239 rows
hotel_prices_raw: 464 rows
rental_prices_raw: 432 rows
selected_travel_options: 24 rows
destination_recommendations: 24 rows


,destination,formatted_address,country,latitude,longitude,google_geocode_place_id,forecast_start_date,forecast_end_date,forecast_days,avg_temp_c,...,selected_accommodation_link,selected_accommodation_data_status,estimated_accommodation_cost,estimated_trip_cost,cost_data_status,weather_score,cost_score,score_without_cost,cost_score_for_model,final_recommendation_score
13,"Lisbon, Portugal","Lisbon, Portugal",Portugal,38.722252,-9.139337,ChIJO_PkYRozGQ0R0DaQ5L3rAAQ,2026-06-26,2026-07-01,6,23.0,...,https://www.thegreenviewhome.com/,accommodation available,142.0,283.0,flight and accommodation available,100,100.0,90.3,100.0,92.8
21,"Budapest, Hungary","Budapest, Hungary",Hungary,47.497912,19.040235,ChIJyc_U0TTDQUcRYBEeDCnEAAQ,2026-06-26,2026-07-01,6,31.0,...,https://www.booking.com/hotel/hu/l1-city-centr...,accommodation available,215.0,340.0,flight and accommodation available,95,95.2,91.5,95.2,92.4
12,"Barcelona, Spain","Barcelona, Spain",Spain,41.387437,2.168650,ChIJ5TCOcRaYpBIRCmZHTz37sEQ,2026-06-26,2026-07-01,6,28.1,...,https://www.bluepillow.com/search/65ca82af671b...,accommodation available,238.0,319.0,flight and accommodation available,100,97.0,90.1,97.0,91.9
10,"Istanbul, Türkiye","Istanbul, İstanbul, Türkiye",Türkiye,41.008238,28.978359,ChIJawhoAASnyhQR0LABvJj-zOE,2026-06-26,2026-07-01,6,26.1,...,https://www.booking.com/hotel/tr/modern-compac...,accommodation available,143.0,390.0,flight and accommodation available,100,91.1,91.8,91.1,91.7
4,"Rome, Italy","Rome, Metropolitan City of Rome Capital, Italy",Italy,41.896707,12.482202,ChIJu46S-ZZhLxMROG5lkwZ3D7k,2026-06-26,2026-07-01,6,29.9,...,https://www.salottotiburtino.it/,accommodation available,237.0,351.0,flight and accommodation available,100,94.3,90.7,94.3,91.7


## 9. Save only the useful CSV outputs

In [89]:
destination_recommendations_path = PROCESSED_DIR / "destination_recommendations.csv"
places_path = PROCESSED_DIR / "places.csv"
weather_daily_path = PROCESSED_DIR / "weather_daily.csv"
selected_travel_options_path = PROCESSED_DIR / "selected_travel_options.csv"

destination_recommendations_df.to_csv(destination_recommendations_path, index=False)
results["places"].to_csv(places_path, index=False)
results["weather_daily"].to_csv(weather_daily_path, index=False)
selected_travel_options_df.to_csv(selected_travel_options_path, index=False)

print(f"Saved: {destination_recommendations_path}")
print(f"Saved: {places_path}")
print(f"Saved: {weather_daily_path}")
print(f"Saved: {selected_travel_options_path}")

Saved: data\processed\destination_recommendations.csv
Saved: data\processed\places.csv
Saved: data\processed\weather_daily.csv
Saved: data\processed\selected_travel_options.csv


## 10. Quick check table

In [91]:
cols = [
    "destination", "country", "latitude", "longitude",
    "forecast_start_date", "forecast_end_date", "avg_temp_c", "peak_max_temp_c",
    "avg_humidity_pct", "avg_cloudiness_pct", "avg_wind_speed_mps", "dominant_weather_description",
    "restaurants_count", "bars_count", "nightclubs_count", "hotels_count", "attractions_count",
    "selected_flight_price", "selected_accommodation_name", "selected_accommodation_type",
    "selected_accommodation_nightly_price", "estimated_accommodation_cost", "estimated_trip_cost", "cost_data_status",
    "weather_score", "nightlife_score", "amenity_score", "cost_score", "final_recommendation_score"
]
cols = [c for c in cols if c in destination_recommendations_df.columns]
display(destination_recommendations_df[cols])
print("Missing values:")
display(destination_recommendations_df[cols].isna().sum())

,destination,country,latitude,longitude,forecast_start_date,forecast_end_date,avg_temp_c,peak_max_temp_c,avg_humidity_pct,avg_cloudiness_pct,...,selected_accommodation_type,selected_accommodation_nightly_price,estimated_accommodation_cost,estimated_trip_cost,cost_data_status,weather_score,nightlife_score,amenity_score,cost_score,final_recommendation_score
13,"Lisbon, Portugal",Portugal,38.722252,-9.139337,2026-06-26,2026-07-01,23.0,32.5,63.7,25.9,...,hotel,20.0,142.0,283.0,flight and accommodation available,100,80.8,88.0,100.0,92.8
21,"Budapest, Hungary",Hungary,47.497912,19.040235,2026-06-26,2026-07-01,31.0,40.9,36.3,31.2,...,hotel,31.0,215.0,340.0,flight and accommodation available,95,88.8,89.7,95.2,92.4
12,"Barcelona, Spain",Spain,41.387437,2.168650,2026-06-26,2026-07-01,28.1,31.0,53.5,17.8,...,hotel,34.0,238.0,319.0,flight and accommodation available,100,80.7,87.6,97.0,91.9
10,"Istanbul, Türkiye",Türkiye,41.008238,28.978359,2026-06-26,2026-07-01,26.1,31.0,61.2,0.1,...,hotel,20.0,143.0,390.0,flight and accommodation available,100,84.2,89.2,91.1,91.7
4,"Rome, Italy",Italy,41.896707,12.482202,2026-06-26,2026-07-01,29.9,38.9,47.8,23.6,...,hotel,34.0,237.0,351.0,flight and accommodation available,100,81.2,89.0,94.3,91.7
5,"Marrakech, Morocco",Morocco,31.622522,-7.989826,2026-06-26,2026-07-01,31.1,42.8,31.4,43.6,...,rental,22.0,153.0,299.0,flight and accommodation available,90,88.4,84.6,98.7,90.7
3,"Paris, France",France,48.857548,2.351377,2026-06-26,2026-07-01,26.2,37.7,44.3,42.9,...,hotel,47.0,326.0,415.0,flight and accommodation available,100,83.1,86.9,89.0,90.4
1,"Dubai, United Arab Emirates",United Arab Emirates,25.204849,55.270783,2026-06-26,2026-07-01,32.4,36.4,64.2,2.8,...,hotel,21.0,148.0,743.0,flight and accommodation available,100,89.1,90.4,61.6,85.8
17,"Hurghada, Egypt",Egypt,27.257896,33.811607,2026-06-26,2026-07-01,31.8,35.0,31.5,0.0,...,rental,20.0,140.0,542.0,flight and accommodation available,100,75.5,83.9,78.4,85.3
23,"Abu Dhabi, United Arab Emirates",United Arab Emirates,24.453884,54.377344,2026-06-26,2026-07-01,32.4,34.3,65.0,2.1,...,rental,30.0,210.0,764.0,flight and accommodation available,100,85.9,88.8,59.8,84.2


Missing values:


destination                             0
country                                 0
latitude                                0
longitude                               0
forecast_start_date                     0
forecast_end_date                       0
avg_temp_c                              0
peak_max_temp_c                         0
avg_humidity_pct                        0
avg_cloudiness_pct                      0
avg_wind_speed_mps                      0
dominant_weather_description            0
restaurants_count                       0
bars_count                              0
nightclubs_count                        0
hotels_count                            0
attractions_count                       0
selected_flight_price                   0
selected_accommodation_name             0
selected_accommodation_type             0
selected_accommodation_nightly_price    0
estimated_accommodation_cost            0
estimated_trip_cost                     0
cost_data_status                  